In [1]:
# T0–T2: referência, aritmética exata e desempenho real da A100
import subprocess, sys, json, time
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],capture_output=True,text=True).stdout)
import experiment as ex
release = ex.prepare_release()
gpu = ex.GPU()
ex.validate_gpu(gpu)
bench = ex.benchmark(gpu)
print(json.dumps(bench,indent=2))
print('T0–T2 CONCLUIDOS; GPU validada contra CPU inteira.')


NVIDIA A100-SXM4-40GB, 40960 MiB

T0_integrity.json saved
T1_gpu_validation.json saved
T2_benchmark.json saved
T2_benchmark.json saved
T2_benchmark.json saved
T2_benchmark.json saved
T2_benchmark.json saved
T2_benchmark.json saved
T2_benchmark.json saved
T2_benchmark.json saved
T2_benchmark.json saved
[
  {
    "n": 64,
    "batch": 256,
    "cpu_seconds": 0.0003768469999840818,
    "gpu_resident_seconds": 9.11599999540158e-05,
    "gpu_including_transfers_seconds": 0.00024492899996175765,
    "speedup_including_transfers": 1.5385968996848938
  },
  {
    "n": 64,
    "batch": 4096,
    "cpu_seconds": 0.006908307999992758,
    "gpu_resident_seconds": 9.940599994706645e-05,
    "gpu_including_transfers_seconds": 0.0005732390000048326,
    "speedup_including_transfers": 12.05135728716036
  },
  {
    "n": 64,
    "batch": 32768,
    "cpu_seconds": 0.07634881600000654,
    "gpu_resident_seconds": 0.00022289600002523002,
    "gpu_including_transfers_seconds": 0.0044093889999885505,
    "sp

In [2]:
# T3 em processo CPU separado; T6–T7 na A100 em paralelo
import experiment_science as ex
from pathlib import Path
import subprocess, sys, json, time
gpu = ex.GPU()
ex.validate_gpu(gpu)
ex.validate_poisson()
reference_log = open(ex.ROOT/'T3_process.log','w')
reference_job = subprocess.Popen([sys.executable,'-u','experiment_science.py','--capture',str(release)],stdout=reference_log,stderr=subprocess.STDOUT)
print('Referencia CPU iniciada, PID:',reference_job.pid,flush=True)
for n in (12,16,20):
    ranks=ex.inspect_images(n)
    print('POSTOS',n,[(r['name'],r['rank']) for r in ranks],flush=True)
for n,name in [(12,'joint_period2'),(16,'gJ'),(16,'galt'),(16,'joint_ends'),(16,'joint_period2'),(20,'gJ'),(20,'galt'),(20,'joint_ends')]:
    print('ENUMERACAO EXATA',n,name,flush=True)
    ex.enumerate_image_gpu(gpu,n,name,max_rank=24,batch=8192)
    print((ex.ROOT/f'T7_n{n}_{name}.json').read_text(),flush=True)
print('T6–T7 CONCLUIDOS. Status da referencia:',reference_job.poll(),flush=True)


T1_gpu_validation.json saved
T7_poisson_validation.json saved
Referencia CPU iniciada, PID: 2920
T6_n12_ranks.json saved
POSTOS 12 [('g0', 3), ('gJ', 5), ('galt', 6), ('joint_ends', 8), ('joint_period2', 12)]
T6_n16_ranks.json saved
POSTOS 16 [('g0', 4), ('gJ', 8), ('galt', 14), ('joint_ends', 12), ('joint_period2', 21)]
T6_n20_ranks.json saved
POSTOS 20 [('g0', 5), ('gJ', 13), ('galt', 22), ('joint_ends', 18), ('joint_period2', 32)]
ENUMERACAO EXATA 12 joint_period2
T7_n12_joint_period2_progress.json saved
T7_n12_joint_period2.json saved
{
  "status": "EXACT_COMPLETE_IMAGE",
  "n": 12,
  "rank": 12,
  "image_functions": 4096,
  "fibers": [
    0,
    21,
    42,
    63
  ],
  "frequencies_histogram": [
    0,
    21,
    63
  ],
  "common_spectral_gcd": 8,
  "zero_A_count_over_fibers": 3456,
  "minimum_abs_A": 0,
  "seconds_this_session": 3.08096199900001,
  "partial_energy_survivors": 3174,
  "poisson_lineJ_survivors": 1504,
  "combined_survivors": 1382,
  "coefficient_coset_size": 2

In [3]:
# T8: confirmação CPU integral e classificação dos setores
import sector_analysis as sa
sectors = sa.run(gpu)
print((ex.ROOT/'T8_n20_complement_CPU.json').read_text())
print((ex.ROOT/'T8_n20_sector_law.json').read_text())
# T9: certificado por interseções (inclusão-exclusão), não por amostragem
for n,k in [(12,3),(16,3),(20,4),(20,5)]:
    print(json.dumps(sa.ward_certificate(n,k),indent=2))
print('Referencia:',reference_job.poll())
print((ex.ROOT/'T3_process.log').read_text()[-1600:])


T8_n20_complement_CPU.json saved
T8_n20_sector_law.json saved
{
  "status": "PASS",
  "complete_image_functions": 8192,
  "full_frequency_coefficients_checked": 8388608,
  "histograms_match_GPU": true,
  "common_spectral_gcd": 32,
  "minimum_abs_A": 64,
  "seconds": 0.832758302000002
}

{
  "status": "EXACT_FINITE_CLASSIFICATION",
  "short_orbit_indices": [
    100,
    186,
    228,
    242,
    244
  ],
  "short_orbit_supports": [
    [
      0,
      1,
      10,
      11
    ],
    [
      0,
      2,
      10,
      12
    ],
    [
      0,
      3,
      10,
      13
    ],
    [
      0,
      4,
      10,
      14
    ],
    [
      0,
      5,
      10,
      15
    ]
  ],
  "sectors": [
    {
      "mask": 0,
      "short_coefficients": [
        0,
        0,
        0,
        0,
        0
      ],
      "Wg0_J": 0,
      "square_mod80": 0,
      "condition_mod80": false,
      "survivors_out_of_8192": 0
    },
    {
      "mask": 1,
      "short_coefficients": [
        1,

In [4]:
# T4–T5: todas as frequências de Walsh, sem transporte entre fibras
assert reference_job.poll() == 0, (ex.ROOT/'T3_process.log').read_text()[-2000:]
fail = ex.audit_n12_gpu(gpu,batch=512)
ex.greedy_certificate(fail)
print((ex.ROOT/'T4_full_94544_gpu.json').read_text())
print((ex.ROOT/'T5_frequency_cover.json').read_text())


T4_progress.json saved
T4_progress.json saved
T4_progress.json saved
T4_progress.json saved
T4_progress.json saved
T4_progress.json saved
T4_progress.json saved
T4_progress.json saved
T4_progress.json saved
T4_progress.json saved
T4_full_94544_gpu.json saved
T5_frequency_cover.json saved
{
  "status": "PASS",
  "complete_functions": 94544,
  "energy_counts": [
    32176,
    39520,
    5552,
    0
  ],
  "full_4096_Walsh_bent_count": 0,
  "cpu_first_batch_comparison": "exact entire arrays",
  "examples": [
    {
      "coefficient_mask": 6352946238516,
      "bad_frequency": 0,
      "W": 224
    },
    {
      "coefficient_mask": 6400174364342,
      "bad_frequency": 0,
      "W": 32
    },
    {
      "coefficient_mask": 1525238077944,
      "bad_frequency": 0,
      "W": -32
    },
    {
      "coefficient_mask": 1606825417594,
      "bad_frequency": 0,
      "W": -32
    },
    {
      "coefficient_mask": 1279890108904,
      "bad_frequency": 1,
      "W": 160
    },
    {
      "c

In [5]:
# T10: controles adversariais adicionais e preservação das evidências
import numpy as np, hashlib, zipfile, platform
gs20=ex.generators(20); ff20=ex.fibers(20,[0,1023],gs20)
short20=[i for i,(_,orb) in enumerate(gs20) if len(orb)<20]
assert not np.any(ff20[[i for i in range(245) if i not in short20],0,:])
gs16=ex.generators(16); f16=ex.fibers(16,[255],gs16)[:,0,:]
b16,p16,piv16=ex.rref(f16); tt16=sa.span_rows(b16)
w16=ex.fwht_cpu(1-2*tt16.astype(np.int32))
first=int(np.flatnonzero(w16[:,0]==0)[0])
inds=[i for i in range(len(b16)) if first>>i&1]
c16=np.bitwise_xor.reduce(p16[inds],axis=0)
assert int(np.sum(1-2*((c16.astype(np.int64)@f16)&1)))==0
ex.save('T10_counterexample_n16.json',{'status':'PASS','A_J':0,'meaning':'Refutes unconditional nonzero complementary sum in every even dimension; NOT a bent counterexample','coefficient_orbit_indices':np.flatnonzero(c16).tolist(),'orbit_supports':[list(gs16[i][0]) for i in np.flatnonzero(c16)]})
law=json.loads((ex.ROOT/'T8_n20_sector_law.json').read_text())
assert law['all_or_none_in_each_sector'] and law['mod80_matches_nonempty_sectors']
assert law['necessary_sector_indicator_ANF_masks']==[2,8,18,24]
ex.save('T10_summary.json',{'status':'PASS','gpu':'NVIDIA A100-SXM4-40GB','n12_full_Walsh_functions':94544,'n12_bent_count':0,'n20_sector_conditions':['c5=0','c2 XOR c4=1'],'n20_excluded_fraction':'3/4','n20_complement_Walsh_divisor':32,'n20_min_abs_A_J':64,'stronger_divisor_64':'REFUTED by A_J=96','n20_closed':False,'n16_closed':False})
ex.save('runtime.json',{'python':sys.version,'numpy':np.__version__,'cupy':gpu.cp.__version__,'cuda_runtime':gpu.cp.cuda.runtime.runtimeGetVersion(),'cuda_driver':gpu.cp.cuda.runtime.driverGetVersion(),'platform':platform.platform()})
archive=Path('/content/RESULTADOS_GRAU4_A100_EXECUTADOS.zip')
entries=[]
for p in sorted(ex.ROOT.rglob('*')):
    if p.is_file() and 'release_original' not in p.parts:
        entries.append((p,'results/'+str(p.relative_to(ex.ROOT))))
for filename in ('experiment.py','experiment_science.py','sector_analysis.py'):
    entries.append((Path(filename),'code/'+filename))
entries.append((Path('Pacote_Final_Grau4_n12.zip'),'reference/Pacote_Final_Grau4_n12.zip'))
manifest=''.join(hashlib.sha256(p.read_bytes()).hexdigest()+'  '+name+'\n' for p,name in entries)
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p,name in entries:z.write(p,name)
    z.writestr('SHA256SUMS.txt',manifest)
with zipfile.ZipFile(archive) as z:assert z.testzip() is None
print('RESULTADOS CONFIRMADOS NA A100')
print('n=12: Walsh completo de 94.544 funções, zero bent.')
print('n=20: 75% excluídos por condições necessárias; caso NÃO fechado.')
print('n=20: 32 divide W_gJ(b); mínimo |A_J|=64; hipótese de divisor 64 refutada.')
print('Certificados, código, checkpoints e logs preservados em',archive)
print('ZIP bytes:',archive.stat().st_size)


T10_counterexample_n16.json saved
T10_summary.json saved
runtime.json saved
RESULTADOS CONFIRMADOS NA A100
n=12: Walsh completo de 94.544 funções, zero bent.
n=20: 75% excluídos por condições necessárias; caso NÃO fechado.
n=20: 32 divide W_gJ(b); mínimo |A_J|=64; hipótese de divisor 64 refutada.
Certificados, código, checkpoints e logs preservados em /content/RESULTADOS_GRAU4_A100_EXECUTADOS.zip
ZIP bytes: 2187838


In [6]:
# Exportação final com conferência de cada hash
manifest=''.join(hashlib.sha256(p.read_bytes()).hexdigest()+'  '+name+chr(10) for p,name in entries)
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p,name in entries:z.write(p,name)
    z.writestr('SHA256SUMS.txt',manifest)
with zipfile.ZipFile(archive) as z:
    assert z.testzip() is None
    lines=z.read('SHA256SUMS.txt').decode().splitlines()
    assert len(lines)==len(entries)
    for line in lines:
        sha,name=line.split('  ',1)
        assert hashlib.sha256(z.read(name)).hexdigest()==sha
print('PASS: arquivo e todos os hashes conferidos. Download das evidências abaixo.')
from google.colab import files
files.download(str(archive))


PASS: arquivo e todos os hashes conferidos. Download das evidências abaixo.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
import os, importlib
print('CWD',os.getcwd(),'FILES',os.listdir('/content'))
importlib.invalidate_caches()
# T11: Poisson no plano, validação integral n12 e piloto n20
import period2_exhaustive as p2
p2.validate_plane()
s12=p2.Sieve(gpu,12)
print('CPU n12:',s12.cpu_check(np.arange(4096)))
print(s12.run())
del s12
gpu.cp.get_default_memory_pool().free_all_blocks()
s20=p2.Sieve(gpu,20)
rng=np.random.default_rng(2026101602)
print('CPU n20:',s20.cpu_check(rng.integers(0,1<<30,256,dtype=np.uint64)))
ex.save('T11_gpu_lookup_validation.json',{'status':'PASS','n12_all_points':4096,'n20_random_points':256,'oracle':'Original ANF CPU, all fibers and all frequencies'})
print(s20.run(limit=1<<24))
print('PILOTO CONCLUIDO; ENUMERACAO TOTAL AINDA PENDENTE')

CWD /content FILES ['.config', 'experiment.py', 'experiment_science.py', 'sector_analysis.py', 'RESULTADOS_GRAU4_A100_EXECUTADOS.zip', 'period2_exhaustive.py', 'Pacote_Final_Grau4_n12.zip', '__pycache__', 'grau4_a100_results', 'sample_data']
T11_plane_validation.json saved
TABLE 12 0 rank 3 bytes 1024
TABLE 12 1 rank 6 bytes 8192
TABLE 12 2 rank 6 bytes 8192
TABLE 12 3 rank 5 bytes 4096
CPU n12: 361
T11_n12_progress.json saved
T11_n12_result.json saved
{'status': 'EXACT_COMPLETE_IMAGE', 'n': 12, 'full_image_rank': 12, 'sector_restricted_rank': 12, 'enumerated': 4096, 'target': 4096, 'survivors': 361, 'table_build_seconds': 0.32790174100000513, 'session_seconds': 0.46752398600028755, 'filters': ['prior necessary sector law for n20', 'four-fiber partial energy', 'line-J Poisson', 'plane Poisson with cyclic dual-sign grouping'], 'scope': 'Necessary conditions only. Survivors are not asserted bent. A prefix pilot is not an exhaustive classification.', 'surviving_fraction_original': '361/40

In [9]:
# T11 completo: cobertura integral do mapa de posto 32 após a lei de setores
result20=s20.run()
print(json.dumps(result20,indent=2))
ck20=np.load(ex.ROOT/'T11_n20_checkpoint.npz')
print('AUDITORIA CPU DOS SOBREVIVENTES:',s20.cpu_check(ck20['examples']))
assert result20['status']=='EXACT_COMPLETE_IMAGE'
print('ENUMERACAO COMPLETA; SOBREVIVENTES NAO SAO DECLARADOS BENT')

T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.json saved
T11_n20_progress.jso

In [10]:
# T12: identificar uma lei afim candidata, ainda sem promovê-la a teorema
cp=gpu.cp
probe=np.random.default_rng(9921).integers(0,1<<30,8192,dtype=np.uint64)
accepted=probe[cp.asnumpy(s20.evaluate(probe))==0]
bits=((accepted[:512,None]>>np.arange(30,dtype=np.uint64))&1).astype(np.uint8)
rb,_,pivs=ex.rref(bits[1:]^bits[0])
freecols=[j for j in range(30) if j not in pivs]
candidate_laws=[]
for j in freecols:
    v=np.zeros(30,dtype=np.uint8);v[j]=1;v[pivs]=rb[:,j]
    mask=sum(int(b)<<i for i,b in enumerate(v));rhs=int((v@bits[0])&1)
    candidate_laws.append((mask,rhs))
print('RANK AFIN DOS SOBREVIVENTES AMOSTRADOS:',len(rb))
print('LEIS CANDIDATAS EM COORDENADAS LIVRES:',candidate_laws)
print('HISTOGRAMA PRIMEIRA FREQUENCIA REJEITANTE:',[(int(cp.asnumpy(s20.order[i-1])),int(c)) for i,c in enumerate(ck20['counts']) if i and c])
ex.save('T12_candidate_laws.json',{'status':'SAMPLE_ONLY','sample':len(probe),'accepted':len(accepted),'affine_rank':len(rb),'candidate_laws':candidate_laws})

RANK AFIN DOS SOBREVIVENTES AMOSTRADOS: 29
LEIS CANDIDATAS EM COORDENADAS LIVRES: [(32913392, 1)]
HISTOGRAMA PRIMEIRA FREQUENCIA REJEITANTE: [(0, 536877056)]
T12_candidate_laws.json saved


In [11]:
# T12: verificar a lei em TODOS os estados e exportar todas as exceções
importlib.invalidate_caches()
import period2_structure as ps
structure20=ps.verify(s20,*candidate_laws[0])
print(json.dumps(structure20,indent=2))
# Controle adicional independente no caso n16
s16=p2.Sieve(gpu,16)
print('CPU n16:',s16.cpu_check(np.random.default_rng(16).integers(0,1<<21,256,dtype=np.uint64)))
print(json.dumps(s16.run(),indent=2))
print('T11 E T12 COMPLETOS')

T12_verified_structure.json saved
{
  "status": "EXACT_COMPLETE_IMAGE",
  "n": 20,
  "states": 1073741824,
  "counts_reject_outside_reject_inside_accept_outside_accept_inside": [
    536870912,
    6144,
    0,
    536864768
  ],
  "necessary_affine_law_original_indices": [
    1,
    2,
    3,
    5,
    6,
    7,
    9,
    10,
    11,
    13,
    14,
    15,
    16,
    17,
    19,
    20,
    21,
    23,
    24,
    25,
    27,
    28,
    29,
    32,
    33,
    36,
    37,
    40,
    41,
    44,
    45,
    46,
    49,
    50,
    53,
    54,
    57,
    59,
    60,
    61,
    63,
    64,
    65,
    67,
    68,
    69,
    70,
    71,
    73,
    74,
    75,
    77,
    78,
    79,
    82,
    83,
    86,
    87,
    90,
    91,
    92,
    95,
    96,
    99,
    101,
    102,
    103,
    105,
    106,
    107,
    108,
    110,
    111,
    112,
    114,
    115,
    118,
    119,
    122,
    123,
    124,
    125,
    126,
    128,
    130,
    132,
    134,
    136,
    

In [13]:
# T13: fórmulas de todos os geradores em seis dimensões e base de 4 variáveis
importlib.invalidate_caches()
import folding_prime_certificate as fp
print(fp.run())
fold=np.load(ex.ROOT/'T13_n20_fold_map.npz')['generator_fold_anf']
exception=np.load(ex.ROOT/'T12_affine_exceptions.npz')
assert np.array_equal(exception['original_linear_form'],fold[:,7])
assert int(exception['original_rhs'])==0
print('LEI AFIM IDENTIFICADA EXATAMENTE COMO COEFICIENTE CUBICO DA RESTRICAO')
exceptional=exception['exceptional_ids']
for lo in range(0,len(exceptional),256):
    assert s20.cpu_check(exceptional[lo:lo+256])==0
ex.save('T13_exception_oracle.json',{'status':'PASS','all_exceptional_states_checked_on_CPU':len(exceptional),'original_ANF_all_fibers_all_frequencies':True,'linear_law_is_folded_cubic_coefficient':True})
print('TODAS AS 6144 EXCECOES CONFERIDAS NA CPU')

T13_prime_folding.json saved
[{'p': 5, 'n': 20, 'quartic_generators': 245, 'all_generator_foldings_match': True, 'folding_rank': 5, 'necessary_equations_rank': 3, 'fraction_satisfying_three_conditions': '1/8'}, {'p': 7, 'n': 28, 'quartic_generators': 735, 'all_generator_foldings_match': True, 'folding_rank': 5, 'necessary_equations_rank': 3, 'fraction_satisfying_three_conditions': '1/8'}, {'p': 11, 'n': 44, 'quartic_generators': 3091, 'all_generator_foldings_match': True, 'folding_rank': 5, 'necessary_equations_rank': 3, 'fraction_satisfying_three_conditions': '1/8'}, {'p': 13, 'n': 52, 'quartic_generators': 5213, 'all_generator_foldings_match': True, 'folding_rank': 5, 'necessary_equations_rank': 3, 'fraction_satisfying_three_conditions': '1/8'}, {'p': 17, 'n': 68, 'quartic_generators': 11985, 'all_generator_foldings_match': True, 'folding_rank': 5, 'necessary_equations_rank': 3, 'fraction_satisfying_three_conditions': '1/8'}, {'p': 19, 'n': 76, 'quartic_generators': 16891, 'all_gener

In [14]:
# Exportação da continuação: resultados T0–T13 e fontes realmente executadas
from fractions import Fraction
final_summary={'status':'PASS','n20_image_rank':32,'n20_states_enumerated_after_prior_sector_filter':1<<30,'n20_survivors':536864768,'n20_excluded_fraction':str(1-Fraction(536864768,1<<32)),'n20_excluded_percent':float(100*(1-Fraction(536864768,1<<32))),'all_6144_exceptions_CPU_checked':True,'n16_survivors':391804,'n16_closed':False,'n20_closed':False,'analytic_family':'n=4p, prime p>=5: three independent necessary linear equations exclude 7/8; proof supplied separately; novelty not established'}
ex.save('T13_final_summary.json',final_summary)
archive2=Path('/content/GRAU4_A100_CONTINUACAO_T11_T13.zip')
entries2=[(p,'results/'+str(p.relative_to(ex.ROOT))) for p in sorted(ex.ROOT.rglob('*')) if p.is_file() and 'release_original' not in p.parts]
for filename in ['experiment.py','experiment_science.py','sector_analysis.py','period2_exhaustive.py','period2_structure.py','folding_prime_certificate.py']:
    entries2.append((Path(filename),'code/'+filename))
entries2.append((Path('Pacote_Final_Grau4_n12.zip'),'reference/Pacote_Final_Grau4_n12.zip'))
manifest2=''.join(hashlib.sha256(p.read_bytes()).hexdigest()+'  '+name+chr(10) for p,name in entries2)
with zipfile.ZipFile(archive2,'w',zipfile.ZIP_DEFLATED) as z:
    for p,name in entries2:z.write(p,name)
    z.writestr('SHA256SUMS.txt',manifest2)
with zipfile.ZipFile(archive2) as z:
    assert z.testzip() is None
    for line in z.read('SHA256SUMS.txt').decode().splitlines():
        sha,name=line.split('  ',1);assert hashlib.sha256(z.read(name)).hexdigest()==sha
print('A100: CONTINUACAO CONCLUIDA E HASHES CONFERIDOS')
print('n20: 1.073.741.824 estados enumerados; exclusao 87,500143%; caso aberto.')
print('6144 excecoes conferidas na CPU. Prova estrutural para n=4p, p primo>=5, em adendo.')
print(archive2,archive2.stat().st_size)
files.download(str(archive2))

T13_final_summary.json saved
A100: CONTINUACAO CONCLUIDA E HASHES CONFERIDOS
n20: 1.073.741.824 estados enumerados; exclusao 87,500143%; caso aberto.
6144 excecoes conferidas na CPU. Prova estrutural para n=4p, p primo>=5, em adendo.
/content/GRAU4_A100_CONTINUACAO_T11_T13.zip 2235740


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>